## Week 06 — Module 17

Gaussian Processes, neural-network comparison, and historical selection performance.

### Load observations

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Locate the previous week's updated datasets or this module's Data folder.
roots = [Path.cwd(), *Path.cwd().parents]
folders = []
for base in roots:
    folders.extend([
        base / "6. Module 17" / "Data",
        base / "Data",
        base / "5. Module 16" / "updated_data_week_5",
        base / "updated_data_week_5",
        base / "7. Capstone Project" / "6. Module 17" / "Data",
        base / "7. Capstone Project" / "5. Module 16" / "updated_data_week_5",
    ])
folder = next((p for p in folders if all((p / f"function_{i}.csv").is_file() for i in range(1, 9))), None)
if folder is None:
    raise FileNotFoundError("Eight function CSVs not found in Module 17 Data or Week 5 updated data folders. Start Jupyter from your capstone project folder.")
data = {i: pd.read_csv(folder / f"function_{i}.csv") for i in range(1, 9)}
print("Loaded eight datasets:", ", ".join(f"F{i}={len(data[i])}" for i in range(1, 9)))

Loaded eight datasets: F1=15, F2=15, F3=20, F4=35, F5=25, F6=25, F7=35, F8=45


### Week 5 returned observations

In [2]:
week_5_inputs = {1: [0.943533, 0.359421], 2: [0.713092, 0.138155], 3: [0.384686, 0.478261, 0.416295], 4: [0.462451, 0.342242, 0.380323, 0.445636], 5: [0.938942, 0.989452, 0.984552, 0.971823], 6: [0.248909, 0.369411, 0.484746, 0.674732, 0.011701], 7: [0.132842, 0.294043, 0.285122, 0.050957, 0.281982, 0.619768], 8: [0.034135, 0.275291, 0.032179, 0.508682, 0.760754, 0.61795, 0.034127, 0.34942]}
week_5_outputs = {1: -2.4467665937353882e-120, 2: 0.7469622511600147, 3: -0.016451292503935306, 4: -0.6383428250895409, 5: 6691.93167361189, 6: -0.4568735372038142, 7: 1.6447360459041676, 8: 9.742277082966}

### Update observations

In [3]:
previous_bests = {}
for i in range(1, 9):
    input_columns = [c for c in data[i].columns if c != "output"]
    point = np.asarray(week_5_inputs[i], dtype=float)
    observed = data[i][input_columns].to_numpy(dtype=float)
    matches = np.flatnonzero(np.all(np.isclose(observed, point, atol=5e-7, rtol=0), axis=1))
    if len(matches):
        observed_output = float(data[i].iloc[matches[0]]["output"])
        if not np.isclose(observed_output, week_5_outputs[i], rtol=1e-7, atol=1e-9):
            raise ValueError(f"Function {i}: existing point has a different output")
        previous_bests[i] = data[i].drop(index=data[i].index[matches[0]])["output"].max()
    else:
        previous_bests[i] = data[i]["output"].max()
        row = dict(zip(input_columns, point))
        row["output"] = week_5_outputs[i]
        data[i] = pd.concat([data[i], pd.DataFrame([row])], ignore_index=True)

### Week 5 performance

In [4]:
print(pd.DataFrame([{"Function": i, "Previous best": previous_bests[i], "New output": week_5_outputs[i], "Improved": week_5_outputs[i] > previous_bests[i]} for i in range(1, 9)]).to_string(index=False))

 Function  Previous best     New output  Improved
        1   7.710875e-16 -2.446767e-120     False
        2   6.112052e-01   7.469623e-01      True
        3  -2.016691e-02  -1.645129e-02      True
        4  -3.006447e-01  -6.383428e-01     False
        5   4.857450e+03   6.691932e+03      True
        6  -4.585106e-01  -4.568735e-01      True
        7   2.536939e+00   1.644736e+00     False
        8   9.911249e+00   9.742277e+00     False


### Gaussian Process fitting

In [5]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.preprocessing import StandardScaler
gp_models = {}
gp_scalers = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    scaler = StandardScaler()
    y_scaled = scaler.fit_transform(y.reshape(-1, 1)).ravel()
    kernel = RBF(length_scale=0.2, length_scale_bounds=(1e-05, 100000.0))
    gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-06, normalize_y=False, n_restarts_optimizer=10, random_state=42)
    gp.fit(X, y_scaled)
    gp_models[i] = gp
    gp_scalers[i] = scaler

print(pd.DataFrame([{"Function": i, "RBF length scale": float(gp_models[i].kernel_.length_scale)} for i in range(1, 9)]).to_string(index=False))

 Function  RBF length scale
        1          0.000010
        2          0.052087
        3          0.184610
        4          0.423460
        5          0.388822
        6          0.534861
        7          0.247590
        8          1.000680


### GP diagnostics

In [6]:
candidate_points = {}
gp_diagnostics = {}
rng = np.random.default_rng(42)
for i in range(1, 9):
    n_dimensions = len(data[i].columns) - 1
    candidates = rng.random((100000, n_dimensions))
    candidate_points[i] = candidates
    mean_scaled, std_scaled = gp_models[i].predict(candidates, return_std=True)
    scaler = gp_scalers[i]
    mean = scaler.inverse_transform(mean_scaled.reshape(-1, 1)).ravel()
    std = std_scaled * scaler.scale_[0]
    gp_diagnostics[i] = {'mean': mean, 'std': std}

print(pd.DataFrame([{"Function": i, "Mean range": np.ptp(gp_diagnostics[i]["mean"]), "Uncertainty range": np.ptp(gp_diagnostics[i]["std"])} for i in range(1, 9)]).to_string(index=False))

 Function  Mean range  Uncertainty range
        1    0.000635           0.001654
        2    0.831417           0.259242
        3    0.389101           0.078040
        4   34.434331           7.386272
        5 7326.020374        1726.249284
        6    2.348109           0.508036
        7    2.575816           0.664717
        8    4.822485           0.659779


### UCB and Expected Improvement

In [7]:
from scipy.stats import norm
gp_candidate_results = {}
for i in range(1, 9):
    candidates = candidate_points[i]
    mean = gp_diagnostics[i]['mean']
    std = gp_diagnostics[i]['std']
    X_existing = data[i].drop(columns='output').values.astype(float)
    y_existing = data[i]['output'].values.astype(float)
    best_idx = np.argmax(y_existing)
    current_best = y_existing[best_idx]
    best_point = X_existing[best_idx]
    kappa = 1.5
    ucb_scores = mean + kappa * std
    ucb_idx = np.argmax(ucb_scores)
    ucb_point = candidates[ucb_idx]
    improvement = mean - current_best
    with np.errstate(divide='ignore', invalid='ignore'):
        Z = np.divide(improvement, std, out=np.zeros_like(improvement), where=std > 0)
        ei_scores = improvement * norm.cdf(Z) + std * norm.pdf(Z)
    ei_scores[std <= 0] = 0
    ei_idx = np.argmax(ei_scores)
    ei_point = candidates[ei_idx]
    ucb_nearest = np.min(np.linalg.norm(X_existing - ucb_point, axis=1))
    ei_nearest = np.min(np.linalg.norm(X_existing - ei_point, axis=1))
    ucb_from_best = np.linalg.norm(ucb_point - best_point)
    ei_from_best = np.linalg.norm(ei_point - best_point)
    candidate_distance = np.linalg.norm(ucb_point - ei_point)
    gp_candidate_results[i] = {'ucb_point': ucb_point, 'ucb_mean': mean[ucb_idx], 'ucb_std': std[ucb_idx], 'ucb_score': ucb_scores[ucb_idx], 'ei_point': ei_point, 'ei_mean': mean[ei_idx], 'ei_std': std[ei_idx], 'ei_score': ei_scores[ei_idx], 'current_best': current_best, 'best_point': best_point}

print(pd.DataFrame([{"Function": i, "UCB mean": gp_candidate_results[i]["ucb_mean"], "EI mean": gp_candidate_results[i]["ei_mean"], "UCB-EI distance": np.linalg.norm(gp_candidate_results[i]["ucb_point"]-gp_candidate_results[i]["ei_point"])} for i in range(1, 9)]).to_string(index=False))

 Function    UCB mean     EI mean  UCB-EI distance
        1   -0.000635   -0.000635         0.000000
        2    0.644023    0.679240         0.007660
        3   -0.048421   -0.010868         0.369503
        4   -0.455899   -0.278717         0.076555
        5 6213.849027 6213.849027         0.000000
        6   -0.338919   -0.313445         0.144351
        7    2.512543    2.512543         0.000000
        8    9.793025    9.793025         0.000000


### Neural-network ensembles

In [8]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
torch.manual_seed(42)

class SmallNN(nn.Module):

    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(input_dim, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, x):
        return self.network(x)
nn_models = {}
nn_scalers = {}
nn_training_losses = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(np.float32)
    y = data[i]['output'].values.astype(np.float32).reshape(-1, 1)
    y_scaler = StandardScaler()
    y_scaled = y_scaler.fit_transform(y).astype(np.float32)
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y_scaled, dtype=torch.float32)
    models = []
    final_losses = []
    for seed in range(5):
        torch.manual_seed(seed)
        model = SmallNN(input_dim=X.shape[1])
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
        loss_function = nn.MSELoss()
        for epoch in range(2000):
            optimizer.zero_grad()
            predictions = model(X_tensor)
            loss = loss_function(predictions, y_tensor)
            loss.backward()
            optimizer.step()
        models.append(model)
        final_losses.append(loss.item())
    nn_models[i] = models
    nn_scalers[i] = y_scaler
    nn_training_losses[i] = final_losses

print(pd.DataFrame([{"Function": i, "Mean training loss": np.mean(nn_training_losses[i])} for i in range(1, 9)]).to_string(index=False))

 Function  Mean training loss
        1            0.014672
        2            0.023384
        3            0.012198
        4            0.001875
        5            0.000085
        6            0.003526
        7            0.000907
        8            0.001338


### Leave-One-Out Cross-Validation

In [9]:
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_absolute_error
nn_validation_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(np.float32)
    y = data[i]['output'].values.astype(np.float32)
    loo = LeaveOneOut()
    predictions = []
    actuals = []
    baseline_predictions = []
    for train_idx, test_idx in loo.split(X):
        X_train = X[train_idx]
        X_test = X[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]
        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).astype(np.float32)
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
        torch.manual_seed(42)
        model = SmallNN(input_dim=X.shape[1])
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
        loss_function = nn.MSELoss()
        for epoch in range(1000):
            optimizer.zero_grad()
            pred = model(X_train_tensor)
            loss = loss_function(pred, y_train_tensor)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            pred_scaled = model(X_test_tensor).numpy()
        pred_original = y_scaler.inverse_transform(pred_scaled).ravel()[0]
        predictions.append(pred_original)
        actuals.append(y_test[0])
        baseline_predictions.append(np.mean(y_train))
    nn_mae = mean_absolute_error(actuals, predictions)
    baseline_mae = mean_absolute_error(actuals, baseline_predictions)
    beats_baseline = nn_mae < baseline_mae
    nn_validation_results[i] = {'nn_mae': nn_mae, 'baseline_mae': baseline_mae, 'beats_baseline': beats_baseline}

print(pd.DataFrame([{"Function": i, "NN LOOCV MAE": nn_validation_results[i]["nn_mae"], "Baseline MAE": nn_validation_results[i]["baseline_mae"], "Beats baseline": nn_validation_results[i]["beats_baseline"]} for i in range(1, 9)]).to_string(index=False))

 Function  NN LOOCV MAE  Baseline MAE  Beats baseline
        1      0.001675      0.001179           False
        2      0.267443      0.241496           False
        3      0.067227      0.048537           False
        4      1.827993      6.878180            True
        5    273.005550   1375.410321            True
        6      0.309082      0.476405            True
        7      0.323396      0.507559            True
        8      0.294854      0.920553            True


### Neural-network candidates

In [10]:
nn_candidate_results = {}
for i in range(1, 9):
    candidates = candidate_points[i].astype(np.float32)
    candidate_tensor = torch.tensor(candidates, dtype=torch.float32)
    models = nn_models[i]
    scaler = nn_scalers[i]
    model_predictions = []
    for model in models:
        model.eval()
        with torch.no_grad():
            pred_scaled = model(candidate_tensor).numpy()
        pred_original = scaler.inverse_transform(pred_scaled).ravel()
        model_predictions.append(pred_original)
    model_predictions = np.array(model_predictions)
    mean_prediction = np.mean(model_predictions, axis=0)
    model_disagreement = np.std(model_predictions, axis=0)
    best_idx = np.argmax(mean_prediction)
    nn_point = candidates[best_idx]
    nn_predicted_output = mean_prediction[best_idx]
    nn_disagreement = model_disagreement[best_idx]
    X_existing = data[i].drop(columns='output').values.astype(float)
    y_existing = data[i]['output'].values.astype(float)
    best_observation_idx = np.argmax(y_existing)
    best_point = X_existing[best_observation_idx]
    distance_from_best = np.linalg.norm(nn_point - best_point)
    nearest_distance = np.min(np.linalg.norm(X_existing - nn_point, axis=1))
    nn_candidate_results[i] = {'point': nn_point, 'predicted_output': nn_predicted_output, 'disagreement': nn_disagreement, 'distance_from_best': distance_from_best, 'nearest_distance': nearest_distance}

print(pd.DataFrame([{"Function": i, "NN predicted output": nn_candidate_results[i]["predicted_output"], "Model disagreement": nn_candidate_results[i]["disagreement"]} for i in range(1, 9)]).to_string(index=False))

 Function  NN predicted output  Model disagreement
        1             0.003677            0.002351
        2             2.724940            1.237037
        3            -0.001139            0.032445
        4            -0.374836            0.459252
        5          6691.842285           47.407719
        6            -0.406389            0.128387
        7             2.412130            0.419323
        8            10.052735            0.132650


### Model comparison

In [11]:
model_comparison = {}
for i in range(1, 9):
    nn_point = nn_candidate_results[i]['point']
    ucb_point = gp_candidate_results[i]['ucb_point']
    ei_point = gp_candidate_results[i]['ei_point']
    nn_to_ucb = np.linalg.norm(nn_point - ucb_point)
    nn_to_ei = np.linalg.norm(nn_point - ei_point)
    ucb_to_ei = np.linalg.norm(ucb_point - ei_point)
    model_comparison[i] = {'nn_to_ucb': nn_to_ucb, 'nn_to_ei': nn_to_ei, 'ucb_to_ei': ucb_to_ei}

print(pd.DataFrame([{"Function": i, **model_comparison[i]} for i in range(1, 9)]).to_string(index=False))

 Function  nn_to_ucb  nn_to_ei  ucb_to_ei
        1   0.492260  0.492260   0.000000
        2   0.285113  0.290756   0.007660
        3   0.538166  0.260038   0.369503
        4   0.135460  0.150437   0.076555
        5   0.188736  0.188736   0.000000
        6   0.220825  0.205455   0.144351
        7   0.332667  0.332667   0.000000
        8   0.773716  0.773716   0.000000


### Historical method performance

In [12]:
week_5_methods = {1: 'EI / UCB agreement', 2: 'Expected Improvement', 3: 'Expected Improvement', 4: 'Expected Improvement', 5: 'Expected Improvement', 6: 'Neural Network', 7: 'EI / UCB agreement', 8: 'Neural Network'}
historical_performance = []
for i in range(1, 9):
    previous_best = previous_bests[i]
    new_output = week_5_outputs[i]
    improvement = new_output - previous_best
    improved = new_output > previous_best
    historical_performance.append({'Function': i, 'Previous method': week_5_methods[i], 'Previous best': previous_best, 'New output': new_output, 'Improvement': improvement, 'New best?': improved})
historical_performance_df = pd.DataFrame(historical_performance)

print(historical_performance_df.to_string(index=False))

 Function      Previous method  Previous best     New output   Improvement  New best?
        1   EI / UCB agreement   7.710875e-16 -2.446767e-120 -7.710875e-16      False
        2 Expected Improvement   6.112052e-01   7.469623e-01  1.357570e-01       True
        3 Expected Improvement  -2.016691e-02  -1.645129e-02  3.715615e-03       True
        4 Expected Improvement  -3.006447e-01  -6.383428e-01 -3.376982e-01      False
        5 Expected Improvement   4.857450e+03   6.691932e+03  1.834482e+03       True
        6       Neural Network  -4.585106e-01  -4.568735e-01  1.637079e-03       True
        7   EI / UCB agreement   2.536939e+00   1.644736e+00 -8.922030e-01      False
        8       Neural Network   9.911249e+00   9.742277e+00 -1.689723e-01      False


### Candidate neighbourhoods

In [13]:
neighbourhood_results = {}
for i in range(1, 9):
    X_existing = data[i].drop(columns='output').values.astype(float)
    y_existing = data[i]['output'].values.astype(float)
    candidates_to_check = {'UCB': gp_candidate_results[i]['ucb_point'], 'EI': gp_candidate_results[i]['ei_point'], 'NN': nn_candidate_results[i]['point']}
    neighbourhood_results[i] = {}
    for method, point in candidates_to_check.items():
        distances = np.linalg.norm(X_existing - point, axis=1)
        nearest_indices = np.argsort(distances)[:3]
        neighbourhood_results[i][method] = {'point': point, 'distances': distances[nearest_indices], 'outputs': y_existing[nearest_indices]}
        for idx in nearest_indices:
            None

print("Nearest-observation neighbourhoods calculated for all eight functions.")

Nearest-observation neighbourhoods calculated for all eight functions.


### Final selection

In [14]:
final_candidates = {1: {'point': gp_candidate_results[1]['ei_point'], 'method': 'EI / UCB agreement'}, 2: {'point': gp_candidate_results[2]['ei_point'], 'method': 'Expected Improvement'}, 3: {'point': gp_candidate_results[3]['ei_point'], 'method': 'Expected Improvement'}, 4: {'point': gp_candidate_results[4]['ei_point'], 'method': 'Expected Improvement'}, 5: {'point': nn_candidate_results[5]['point'], 'method': 'Neural Network'}, 6: {'point': gp_candidate_results[6]['ei_point'], 'method': 'Expected Improvement'}, 7: {'point': gp_candidate_results[7]['ei_point'], 'method': 'EI / UCB agreement'}, 8: {'point': nn_candidate_results[8]['point'], 'method': 'Neural Network'}}

### Final validation

In [15]:
final_validation = {}
for i in range(1, 9):
    point = np.array(final_candidates[i]['point'], dtype=float)
    method = final_candidates[i]['method']
    X_existing = data[i].drop(columns='output').values.astype(float)
    in_bounds = np.all((point >= 0) & (point <= 1))
    duplicate = np.any(np.all(np.isclose(X_existing, point, atol=1e-08), axis=1))
    distances = np.linalg.norm(X_existing - point, axis=1)
    nearest_distance = np.min(distances)
    final_validation[i] = {'in_bounds': in_bounds, 'duplicate': duplicate, 'nearest_distance': nearest_distance}
all_valid = all((result['in_bounds'] and (not result['duplicate']) for result in final_validation.values()))

print(pd.DataFrame([{"Function": i, "Method": final_candidates[i]["method"], "In bounds": final_validation[i]["in_bounds"], "Duplicate": final_validation[i]["duplicate"], "Nearest distance": final_validation[i]["nearest_distance"]} for i in range(1, 9)]).to_string(index=False))
print("All eight valid:", all_valid)

 Function               Method  In bounds  Duplicate  Nearest distance
        1   EI / UCB agreement       True      False          0.180671
        2 Expected Improvement       True      False          0.030884
        3 Expected Improvement       True      False          0.140675
        4 Expected Improvement       True      False          0.104897
        5       Neural Network       True      False          0.028637
        6 Expected Improvement       True      False          0.213058
        7   EI / UCB agreement       True      False          0.172684
        8       Neural Network       True      False          0.397197
All eight valid: True


### Submission points

In [16]:
submission_points = {}
for i in range(1, 9):
    point = np.array(final_candidates[i]['point'], dtype=float)
    formatted_point = '-'.join((f'{value:.6f}' for value in point))
    submission_points[i] = formatted_point

for i in range(1, 9):
    print(f"F{i}: {submission_points[i]}")

F1: 0.773956-0.438878
F2: 0.729513-0.111999
F3: 0.308260-0.310001-0.479247
F4: 0.329887-0.354827-0.355151-0.453774
F5: 0.939869-0.992261-0.978815-0.999723
F6: 0.397519-0.393747-0.471443-0.711380-0.165385
F7: 0.174885-0.324058-0.479146-0.269144-0.331135-0.614490
F8: 0.042609-0.051269-0.314729-0.127230-0.965221-0.369000-0.150066-0.389171
